<a href="https://colab.research.google.com/github/hunter198333/Digitrades/blob/main_ai_trader_agent/Trading_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You can install Python libraries using the `pip` package installer. For example, to install a library like `numpy`, you would run the following command in a code cell:

```bash
!pip install numpy
```

The `!` before `pip` allows you to run shell commands directly in the Colab notebook. If the library is already installed, `pip` will tell you that the requirement is already satisfied. If you need to install a specific version, you can specify it like this: `!pip install numpy==1.20.0`.

Here's an example:

In [ ]:
!pip install numpy

In [ ]:
import kagglehub
path = kagglehub.dataset_download("borismarjanovic/price-volume-data-for-all-us-stocks-etfs")

100%|██████████| 492M/492M [00:07<00:00, 73.5MB/s]

Extracting files...


In [ ]:
# Install numpy
!pip install numpy

# Verify installation by importing and checking version
import numpy as np
print(f"NumPy version: {np.__version__}")

NumPy version: 2.0.2


In [ ]:
# Install pandas
!pip install pandas

# Verify installation
import pandas as pd
print(f"Pandas version: {pd.__version__}")

Pandas version: 2.2.2


In [ ]:
# Install yfinance library to fetch financial data
!pip install yfinance

In [ ]:
import yfinance as yf
import pandas as pd

# Define the ticker for Bitcoin (BTC-USD) and download daily data
btc_ticker = yf.Ticker("BTC-USD")
btc_data = btc_ticker.history(period="max", interval="1d")

# Display the first 5 rows of the downloaded data
print("Bitcoin daily data:")
display(btc_data.head())

Bitcoin daily data:


,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2014-09-17 00:00:00+00:00,465.864014,468.174011,452.421997,457.334015,21056800,0.0,0.0
2014-09-18 00:00:00+00:00,456.859985,456.859985,413.104004,424.440002,34483200,0.0,0.0
2014-09-19 00:00:00+00:00,424.102997,427.834991,384.532013,394.795990,37919700,0.0,0.0
2014-09-20 00:00:00+00:00,394.673004,423.295990,389.882996,408.903992,36863600,0.0,0.0
2014-09-21 00:00:00+00:00,408.084991,412.425995,393.181000,398.821014,26580100,0.0,0.0


In [ ]:
# Calculate the 50-day Simple Moving Average (SMA)
btc_data['SMA_50'] = btc_data['Close'].rolling(window=50).mean()

print("Bitcoin daily data with 50-day SMA:")
display(btc_data.tail())

Bitcoin daily data with 50-day SMA:


,Open,High,Low,Close,Volume,Dividends,Stock Splits,SMA_50
Date,,,,,,,,
2026-05-27 00:00:00+00:00,75825.304688,76014.296875,74136.500000,74344.703125,33802172927,0.0,0.0,77124.871094
2026-05-28 00:00:00+00:00,74339.570312,74460.125000,72493.414062,73536.554688,40148145327,0.0,0.0,77173.135000
2026-05-29 00:00:00+00:00,73537.031250,74218.562500,72435.625000,73372.523438,34457929250,0.0,0.0,77205.228906
2026-05-30 00:00:00+00:00,73370.851562,74020.757812,73125.234375,73754.835938,19563589191,0.0,0.0,77220.744688
2026-05-31 00:00:00+00:00,73754.984375,74150.984375,73654.382812,73861.429688,17598513152,0.0,0.0,77236.887813


In [ ]:
# Calculate the 20-day Exponential Moving Average (EMA)
btc_data['EMA_20'] = btc_data['Close'].ewm(span=20, adjust=False).mean()

print("Bitcoin daily data with 50-day SMA and 20-day EMA:")
display(btc_data.tail())

Bitcoin daily data with 50-day SMA and 20-day EMA:


,Open,High,Low,Close,Volume,Dividends,Stock Splits,SMA_50,EMA_20
Date,,,,,,,,,
2026-05-27 00:00:00+00:00,75825.304688,76014.296875,74136.500000,74344.703125,33802172927,0.0,0.0,77124.871094,77316.341262
2026-05-28 00:00:00+00:00,74339.570312,74460.125000,72493.414062,73536.554688,40148145327,0.0,0.0,77173.135000,76956.361588
2026-05-29 00:00:00+00:00,73537.031250,74218.562500,72435.625000,73372.523438,34457929250,0.0,0.0,77205.228906,76615.043669
2026-05-30 00:00:00+00:00,73370.851562,74020.757812,73125.234375,73754.835938,19563589191,0.0,0.0,77220.744688,76342.642933
2026-05-31 00:00:00+00:00,73754.984375,74150.984375,73654.382812,73861.429688,17598513152,0.0,0.0,77236.887813,76106.336909


### Define Buy and Sell Signals

We will define buy and sell signals based on the crossover strategy of the `EMA_20` and `SMA_50`:

*   **Buy Signal**: `EMA_20` crosses above `SMA_50`.
*   **Sell Signal**: `EMA_20` crosses below `SMA_50`.

In [ ]:
import yfinance as yf
import pandas as pd

# Define the ticker for Bitcoin (BTC-USD) and download daily data
btc_ticker = yf.Ticker("BTC-USD")
btc_data = btc_ticker.history(period="max", interval="1d")

# Calculate the 50-day Simple Moving Average (SMA)
btc_data['SMA_50'] = btc_data['Close'].rolling(window=50).mean()

# Calculate the 20-day Exponential Moving Average (EMA)
btc_data['EMA_20'] = btc_data['Close'].ewm(span=20, adjust=False).mean()

# Create a 'Signal' column, initialized to 0 (no signal)
btc_data['Signal'] = 0.0

# Generate Buy signals (1.0) when EMA_20 crosses above SMA_50
# This means EMA_20 was below SMA_50 in the previous period and is now above it.
btc_data.loc[btc_data['EMA_20'] > btc_data['SMA_50'], 'Signal'] = 1.0

# Generate Sell signals (-1.0) when EMA_20 crosses below SMA_50
# This means EMA_20 was above SMA_50 in the previous period and is now below it.
btc_data.loc[btc_data['EMA_20'] < btc_data['SMA_50'], 'Signal'] = -1.0

# To detect actual crossovers, we look for the change in signal.
# A buy signal happens when the signal changes from -1.0 to 1.0 (or 0.0 to 1.0 if not already -1.0)
# A sell signal happens when the signal changes from 1.0 to -1.0 (or 0.0 to -1.0 if not already 1.0)

# Calculate the difference to find the actual crossover points
btc_data['Position'] = btc_data['Signal'].diff()

print("Bitcoin daily data with Buy/Sell Signals:")
display(btc_data[btc_data['Position'] != 0].head(10))

Bitcoin daily data with Buy/Sell Signals:


,Open,High,Low,Close,Volume,Dividends,Stock Splits,SMA_50,EMA_20,Signal,Position
Date,,,,,,,,,,,
2014-09-17 00:00:00+00:00,465.864014,468.174011,452.421997,457.334015,21056800,0.0,0.0,NaN,457.334015,0.0,NaN
2014-11-05 00:00:00+00:00,330.683014,343.368988,330.683014,339.485992,19817200,0.0,0.0,372.766840,349.705580,-1.0,-1.0
2014-11-13 00:00:00+00:00,427.273010,457.092987,401.122986,420.734985,58945000,0.0,0.0,365.445421,365.672494,1.0,2.0
2014-12-14 00:00:00+00:00,346.726990,353.316010,345.417999,351.631989,12415200,0.0,0.0,363.239580,362.396580,-1.0,-2.0
2015-02-25 00:00:00+00:00,238.889999,239.339996,235.529999,237.470001,11496200,0.0,0.0,235.776460,236.696480,1.0,2.0
2015-03-31 00:00:00+00:00,247.453995,248.729996,242.738998,244.223999,22672000,0.0,0.0,257.332741,257.305157,-1.0,-2.0
2015-05-22 00:00:00+00:00,235.320999,240.968994,235.059998,240.348007,27003000,0.0,0.0,235.640061,236.228191,1.0,2.0
2015-06-03 00:00:00+00:00,225.735992,227.404007,223.929993,225.873993,17752400,0.0,0.0,233.349380,233.179613,-1.0,-2.0
2015-06-18 00:00:00+00:00,249.427994,252.108002,244.126999,249.007004,30980200,0.0,0.0,235.068240,235.884650,1.0,2.0


In [ ]:
import kagglehub
path = kagglehub.model_download('tensorflow/spam-detection/tensorFlow2/tutorials-spam-detection/1')


  0%|          | 0.00/102k [00:00<?, ?B/s]
100%|██████████| 102k/102k [00:00<00:00, 706kB/s]



100%|██████████| 1.33k/1.33k [00:00<00:00, 1.67MB/s]



  0%|          | 0.00/114k [00:00<?, ?B/s]
100%|██████████| 114k/114k [00:00<00:00, 615kB/s]
